In [1]:
from vllm import LLM, SamplingParams
from tower_eval.utils import read_lines, load_jsonl_file
from pathlib import Path
import itertools
import json

/mnt/data/jpombal/tower-eval/tower-eval-env/lib/python3.10/site-packages/transformers/utils/hub.py:124: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
model_stem = "TowerInstruct-7B-w-chat"
model_name = "/mnt/data/jpombal/axolotl/TowerInstruct_7B_chat_ckpts/checkpoint-5900"
sampling_params = SamplingParams(temperature=0.0, max_tokens=1, prompt_logprobs=1)
model = LLM(model_name)
tokenizer = model.get_tokenizer()

INFO 06-21 13:12:34 llm_engine.py:74] Initializing an LLM engine (v0.4.0.post1) with config: model='/mnt/data/jpombal/axolotl/TowerInstruct_7B_chat_ckpts/checkpoint-5900', tokenizer='/mnt/data/jpombal/axolotl/TowerInstruct_7B_chat_ckpts/checkpoint-5900', tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=False, dtype=torch.bfloat16, max_seq_len=4096, download_dir=None, load_format=auto, tensor_parallel_size=1, disable_custom_all_reduce=True, quantization=None, enforce_eager=False, kv_cache_dtype=auto, device_config=cuda, seed=0)


You are using the default legacy behaviour of the <class 'transformers.models.llama.tokenization_llama_fast.LlamaTokenizerFast'>. This is expected, and simply means that the `legacy` (previous) behavior will be used so nothing changes for you. If you want to use the new behaviour, set `legacy=False`. This should only be set if you understand what it means, and thoroughly read the reason why this was added as explained in https://github.com/huggingface/transformers/pull/24565


INFO 06-21 13:12:35 selector.py:51] Cannot use FlashAttention because the package is not found. Please install it for better performance.
INFO 06-21 13:12:35 selector.py:25] Using XFormers backend.
INFO 06-21 13:12:40 model_runner.py:104] Loading model weights took 12.5532 GB
INFO 06-21 13:12:43 gpu_executor.py:94] # GPU blocks: 7392, # CPU blocks: 512


OutOfMemoryError: CUDA out of memory. Tried to allocate 1.80 GiB. GPU 0 has a total capacty of 79.15 GiB of which 1023.38 MiB is free. Process 1521275 has 42.44 GiB memory in use. Including non-PyTorch memory, this process has 35.71 GiB memory in use. Of the allocated memory 34.22 GiB is allocated by PyTorch, and 15.37 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
root_dir = Path(f"/mnt/data/jpombal/wmt24-chat-translation")
raw_data_dir = root_dir / "raw_data" / "mt"
instructions_dir = root_dir / "instructions"
datasets = ["wmt24_chat_dev"]
lps = [
    "en-de",
    "de-en",
    "en-fr",
    "fr-en",
    "en-ko",
    "ko-en",
    "en-nl",
    "nl-en",
    "en-pt-br",
    "pt-br-en",
]
context_settings = ["no_context", "full_context"]
# run the combination of settings
exp_settings = list(itertools.product(datasets, lps, context_settings))
exp_settings_records = [
    {"dataset": ds, "lp": lp, "context_setting": cs} for ds, lp, cs in exp_settings
]

# On reference

In [ ]:
for e in exp_settings_records:
    print(e)
    instructions_path = (
        instructions_dir
        / e["context_setting"]
        / "mt"
        / f'{e["dataset"]}.{e["lp"]}'
        / "instructions.txt"
    )
    raw_data_path = raw_data_dir / f'{e["dataset"]}.{e["lp"]}' / "test.jsonl"
    # replicate tower-eval data loading
    instructions = read_lines(instructions_path, unescape_newline=True)
    raw_data = load_jsonl_file(raw_data_path)
    references = [r["ref"] for r in raw_data]
    assert len(references) == len(
        instructions
    ), f"Length of instructions and references must be the same; there may be an error upstream."
    # tokenize prompts and references, and join
    tokenized_prompts = []
    instruction_lengths = []
    for inst, ref in zip(instructions, references):
        tok_prompt = tokenizer.encode(inst)
        tok_reference = tokenizer.encode(
            ref,
            add_special_tokens=False,  # do not add bos token because we will merge with prompt; tower instruct doesn't generate a bos token after the translation prompt
        )
        instruction_lengths.append(
            len(tok_prompt)
        )  # register the length of the instruction for later
        tokenized_prompt = tok_prompt + tok_reference
        tokenized_prompts.append(tokenized_prompt)
    output = model.generate(
        prompt_token_ids=tokenized_prompts,
        sampling_params=sampling_params,
        use_tqdm=True,
    )
    ref_log_probs_list = []
    for inst_len, out in zip(instruction_lengths, output):
        # get the log probabilities of each token in the reference.
        # in vllm, log probs are in a list, where each token has a dict with at most two values: the first key is the prompt token id, and respective log prob;
        # if that token is not the model's first token, that dict will have a second key, which is the token id of token with the highest log prob;
        # In this case we only care about the former
        ref_log_probs = [
            d[list(d.keys())[0]].logprob for d in out.prompt_logprobs[inst_len:]
        ]
        ref_log_probs_list.append(ref_log_probs)
    # Save information we care about
    output_dir = (
        root_dir
        / "pcxmi"
        / e["context_setting"]
        / "mt"
        / f'{e["dataset"]}.{e["lp"]}'
        / model_stem
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Writing to {output_dir}...")
    with open(output_dir / "log_probs_ref.jsonl", "w") as f_jsonl, open(
        output_dir / "mean_log_probs_ref.txt", "w"
    ) as f_mean, open(output_dir / "max_log_probs_ref.txt", "w") as f_max, open(
        output_dir / "min_log_probs_ref.txt", "w"
    ) as f_min, open(
        output_dir / "sum_log_probs_ref.txt", "w"
    ) as f_sum:
        for ref_log_probs in ref_log_probs_list:
            # coalesce to zero if no tokens in the reference; happens once in nl-en
            if ref_log_probs == []:
                ref_log_probs = [-0.0]
            json.dump({"log_probs": ref_log_probs}, f_jsonl)
            f_jsonl.write("\n")
            f_mean.write(f"{sum(ref_log_probs) / len(ref_log_probs)}\n")
            f_max.write(f"{max(ref_log_probs)}\n")
            f_min.write(f"{min(ref_log_probs)}\n")
            f_sum.write(f"{sum(ref_log_probs)}\n")

{'dataset': 'wmt24_chat_dev', 'lp': 'en-de', 'context_setting': 'no_context'}


Processed prompts: 100%|██████████| 1301/1301 [00:08<00:00, 149.26it/s]


Writing to /mnt/data/jpombal/wmt24-chat-translation/pcxmi/no_context/mt/wmt24_chat_dev.en-de/TowerInstruct-7B-v0.2...
{'dataset': 'wmt24_chat_dev', 'lp': 'en-de', 'context_setting': 'full_context'}


Processed prompts: 100%|██████████| 1301/1301 [01:28<00:00, 14.77it/s]


Writing to /mnt/data/jpombal/wmt24-chat-translation/pcxmi/full_context/mt/wmt24_chat_dev.en-de/TowerInstruct-7B-v0.2...
{'dataset': 'wmt24_chat_dev', 'lp': 'de-en', 'context_setting': 'no_context'}


...

Writing to /mnt/data/jpombal/wmt24-chat-translation/pcxmi/full_context/mt/wmt24_chat_dev.en-pt-br/TowerInstruct-7B-v0.2...
{'dataset': 'wmt24_chat_dev', 'lp': 'pt-br-en', 'context_setting': 'no_context'}


Processed prompts: 100%|██████████| 1267/1267 [00:08<00:00, 153.45it/s]


Writing to /mnt/data/jpombal/wmt24-chat-translation/pcxmi/no_context/mt/wmt24_chat_dev.pt-br-en/TowerInstruct-7B-v0.2...
{'dataset': 'wmt24_chat_dev', 'lp': 'pt-br-en', 'context_setting': 'full_context'}


Processed prompts: 100%|██████████| 1267/1267 [00:46<00:00, 27.10it/s]

Writing to /mnt/data/jpombal/wmt24-chat-translation/pcxmi/full_context/mt/wmt24_chat_dev.pt-br-en/TowerInstruct-7B-v0.2...


# On hypothesis

In [ ]:
sampling_params = SamplingParams(temperature=0.0, max_tokens=1024, logprobs=1)

In [ ]:
for e in exp_settings_records:
    print(e)
    instructions_path = (
        instructions_dir
        / e["context_setting"]
        / "mt"
        / f'{e["dataset"]}.{e["lp"]}'
        / "instructions.txt"
    )
    raw_data_path = raw_data_dir / f'{e["dataset"]}.{e["lp"]}' / "test.jsonl"
    # replicate tower-eval data loading
    instructions = read_lines(instructions_path, unescape_newline=True)
    output = model.generate(
        instructions,
        sampling_params=sampling_params,
        use_tqdm=True,
    )
    log_probs_list = []
    for out in output:
        # get the log probabilities of each token in the output (hypothesis)
        log_probs = [d[list(d.keys())[0]].logprob for d in out.outputs[0].logprobs]
        log_probs_list.append(log_probs)
    # Save information we care about
    output_dir = (
        root_dir
        / "pcxmi"
        / e["context_setting"]
        / "mt"
        / f'{e["dataset"]}.{e["lp"]}'
        / model_stem
    )
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f"Writing to {output_dir}...")
    with open(output_dir / "log_probs_hyp.jsonl", "w") as f_jsonl, open(
        output_dir / "mean_log_probs_hyp.txt", "w"
    ) as f_mean, open(output_dir / "max_log_probs_hyp.txt", "w") as f_max, open(
        output_dir / "min_log_probs_hyp.txt", "w"
    ) as f_min, open(
        output_dir / "sum_log_probs_hyp.txt", "w"
    ) as f_sum:
        for log_probs in log_probs_list:
            json.dump({"log_probs": log_probs}, f_jsonl)
            f_jsonl.write("\n")
            f_mean.write(f"{sum(log_probs) / len(log_probs)}\n")
            f_max.write(f"{max(log_probs)}\n")
            f_min.write(f"{min(log_probs)}\n")
            f_sum.write(f"{sum(log_probs)}\n")

{'dataset': 'wmt24_chat_dev', 'lp': 'en-de', 'context_setting': 'no_context'}


Processed prompts: 100%|██████████| 1301/1301 [00:10<00:00, 124.71it/s]


Writing to /mnt/data/jpombal/wmt24-chat-translation/pcxmi/no_context/mt/wmt24_chat_dev.en-de/TowerInstruct-7B-v0.2...
{'dataset': 'wmt24_chat_dev', 'lp': 'en-de', 'context_setting': 'full_context'}


Processed prompts:  72%|███████▏  | 935/1301 [00:35<00:11, 31.24it/s]

Processed prompts: 100%|██████████| 1301/1301 [00:40<00:00, 32.34it/s] 


...

Writing to /mnt/data/jpombal/wmt24-chat-translation/pcxmi/full_context/mt/wmt24_chat_dev.en-pt-br/TowerInstruct-7B-v0.2...
{'dataset': 'wmt24_chat_dev', 'lp': 'pt-br-en', 'context_setting': 'no_context'}


Processed prompts: 100%|██████████| 1267/1267 [00:08<00:00, 142.00it/s]


Writing to /mnt/data/jpombal/wmt24-chat-translation/pcxmi/no_context/mt/wmt24_chat_dev.pt-br-en/TowerInstruct-7B-v0.2...
{'dataset': 'wmt24_chat_dev', 'lp': 'pt-br-en', 'context_setting': 'full_context'}


Processed prompts: 100%|██████████| 1267/1267 [00:25<00:00, 49.17it/s] 

Writing to /mnt/data/jpombal/wmt24-chat-translation/pcxmi/full_context/mt/wmt24_chat_dev.pt-br-en/TowerInstruct-7B-v0.2...
